# 1.1 - The Anatomy of a Starting Universe

Notebook for [1.1 - The Anatomy of a Starting Universe](https://thepfplumber.substack.com/p/11-the-anatomy-of-a-starting-universe).

**Rules (TPP defaults)**
* Backtests cover 10 years unless a project says otherwise.
* Public data only.
* Stocks without public prices are skipped later, at the price step.

**Universe definition.** A company is in the starting universe on date \(t\) if all of the following hold:
1. It files US periodic reports (10-K / 10-Q). This excludes foreign 20-F / 40-F filers, the counterpart of the article's ADR exclusion.
2. It shows evidence of a national-exchange listing: a current Nasdaq / NYSE / Cboe ticker, an 8-A12B registration, or a Form 25 delisting.
3. \(t\) falls inside its listing interval \([\text{list\_start}, \text{list\_end})\).
4. It is an operating company, i.e. not a SPAC, commodity pool or asset-backed issuer (by SIC code).

$$U_{starting}(t) = \{ c \in U_{master} : \text{list\_start}_c \le t < \text{list\_end}_c \ \wedge \ \prod_k I_k(c) = 1 \}$$

**Requirements:** Python 3.9+ with `pandas`, `numpy`, `requests` and `pyarrow` (for parquet). Runs in any Jupyter environment or as a plain script.

**Runtime:** about 2-3 minutes for the filing indices, plus about 15-20 minutes for company details on the first run. Set `CACHE_DIR` to reuse both on later runs.

In [0]:
import os, re, time, threading
import concurrent.futures as cf
from datetime import date

import numpy as np
import pandas as pd
import requests
from IPython.display import display

# SEC asks every client to identify itself with a contact in the User-Agent. No account or key is needed.
# Replace with your own name and email before running; SEC rejects requests without a real contact.
USER_AGENT = "Your Name your.email@example.com"
if "example.com" in USER_AGENT:
    raise ValueError("Set USER_AGENT to your own name and email (SEC fair-access policy).")
HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}
MAX_RPS = 8                      # SEC fair-access limit is 10 requests/second; stay below it

# Rolling backtest window (TPP default: 10 years, refreshed on every run)
WINDOW_YEARS = 10
AS_OF = pd.Timestamp(date.today())
WINDOW_START = AS_OF - pd.DateOffset(years=WINDOW_YEARS)
DATA_START = WINDOW_START - pd.DateOffset(months=15)   # extra history so companies alive at WINDOW_START are seen filing

ACTIVE_GRACE = pd.Timedelta(days=200)   # a company counts as active until 200 days after its last 10-K / 10-Q
MAJOR_EXCHANGES = {"Nasdaq", "NYSE", "CBOE"}
EXCLUDED_SIC = {"6770": "Blank checks (SPACs)",
                "6221": "Commodity pools / trusts",
                "6189": "Asset-backed securities"}

PERIODIC_FORMS = {"10-K", "10-K/A", "10-KT", "10-Q", "10-Q/A"}   # US domestic periodic reports
LISTING_FORMS = {"8-A12B"}                                         # registration of a security on a national exchange
DELISTING_FORMS = {"25-NSE", "25"}                                 # removal from listing: by the exchange (25-NSE) or the issuer (25)
FORMS = PERIODIC_FORMS | LISTING_FORMS | DELISTING_FORMS

# Cache and outputs are written to a local folder (relative to the working directory). Change DATA_DIR to save elsewhere.
# Set CACHE_DIR = "" to disable caching (full download every run).
DATA_DIR = "data"
CACHE_DIR = f"{DATA_DIR}/edgar_cache"

print(f"Window: {WINDOW_START.date()} to {AS_OF.date()} (filings read from {DATA_START.date()})")

In [0]:
_lock, _last_call = threading.Lock(), [0.0]

def sec_get(url, as_json=False):
    """GET from sec.gov with the contact header, a shared rate limit and retries."""
    for attempt in range(5):
        with _lock:
            wait = _last_call[0] + 1 / MAX_RPS - time.time()
            if wait > 0:
                time.sleep(wait)
            _last_call[0] = time.time()
        r = requests.get(url, headers=HEADERS, timeout=120)
        if r.status_code == 200:
            return r.json() if as_json else r.text
        if r.status_code == 404:
            return None
        time.sleep(2 ** attempt)        # 429 / 5xx: back off and retry
    r.raise_for_status()

def quarters(start, end):
    """Yield (year, quarter) from the quarter containing start to the one containing end."""
    y, q = start.year, (start.month - 1) // 3 + 1
    while (y, q) <= (end.year, (end.month - 1) // 3 + 1):
        yield y, q
        y, q = (y + 1, 1) if q == 4 else (y, q + 1)

def parse_form_idx(text):
    """Parse an EDGAR full-index form.idx file, keeping only the forms in FORMS."""
    rows, started = [], False
    for line in text.splitlines():
        if not started:
            started = line.startswith("-----")
            continue
        parts = re.split(r"\s{2,}", line.strip())      # columns are separated by 2+ spaces
        if len(parts) == 5 and parts[0] in FORMS:
            rows.append(parts[:4])
    return pd.DataFrame(rows, columns=["form", "name", "cik", "date_filed"])

def load_quarter(y, q):
    """One quarter of filings. Closed quarters never change, so they are cached; the open quarter is always re-read."""
    closed = pd.Timestamp(y, 3 * q, 1) + pd.offsets.MonthEnd(0) < AS_OF
    path = f"{CACHE_DIR}/form_idx_{y}Q{q}.parquet" if CACHE_DIR else None
    if path and closed and os.path.exists(path):
        return pd.read_parquet(path)
    text = sec_get(f"https://www.sec.gov/Archives/edgar/full-index/{y}/QTR{q}/form.idx")
    df = parse_form_idx(text) if text else pd.DataFrame(columns=["form", "name", "cik", "date_filed"])
    if path and closed:
        os.makedirs(CACHE_DIR, exist_ok=True)
        df.to_parquet(path)
    return df

## Step 1: Filing history (master universe source)
EDGAR publishes a quarterly index of every filing. From it we keep three kinds of form:
* **Periodic reports** (10-K, 10-Q): shows the company was an active US reporting company.
* **8-A12B**: a security was registered on a national exchange (listing).
* **Form 25 / 25-NSE**: a security was removed from an exchange (delisting by the issuer or by the exchange).

In [0]:
t0 = time.time()
filings = pd.concat([load_quarter(y, q) for y, q in quarters(DATA_START, AS_OF)], ignore_index=True)
filings["cik"] = filings["cik"].astype(int)
filings["date_filed"] = pd.to_datetime(filings["date_filed"])
filings = filings[filings["date_filed"].between(DATA_START, AS_OF)]

print(f"{len(filings):,} filings, {DATA_START.date()} to {AS_OF.date()}, in {time.time() - t0:.0f}s")
display(filings["form"].value_counts().rename_axis("form").reset_index(name="filings"))

# Today's tickers, used only to tell whether a company is still listed now
tickers_now = pd.DataFrame(sec_get("https://www.sec.gov/files/company_tickers_exchange.json", as_json=True)["data"],
                           columns=["cik", "name", "ticker", "exchange"])
currently_listed = set(tickers_now.loc[tickers_now["exchange"].isin(MAJOR_EXCHANGES), "cik"])
print(f"{len(currently_listed):,} companies with a Nasdaq / NYSE / Cboe ticker today")

## Step 2: Listing intervals
For each company (CIK):
* **list_start**:
  * For a company that first appears inside the data (IPO, spin-off), this is the date of its first 8-A12B.
  * Otherwise it is the first periodic filing in the data, i.e. the company was already listed when the window started.
* **list_end**: the earlier of:
  * its last Form 25 / 25-NSE, if it no longer has a major-exchange ticker today;
  * 200 days after its last 10-K / 10-Q, i.e. it stopped reporting (acquired, bankrupt, deregistered).

  If neither has happened, `list_end` is empty and the company is still listed.
* **status_unknown**: the company is not listed today, has no delisting form in the window, and is still filing. These are excluded.

In [0]:
def first_last(forms):
    g = filings[filings["form"].isin(forms)].groupby("cik")["date_filed"]
    return g.min(), g.max()

per_first, per_last = first_last(PERIODIC_FORMS)
reg_first, _ = first_last(LISTING_FORMS)
_, delist_last = first_last(DELISTING_FORMS)

co = pd.DataFrame({"first_periodic": per_first, "last_periodic": per_last})   # periodic filers only
co.index.name = "cik"
co["first_8a12b"] = reg_first.reindex(co.index)
co["last_delisting"] = delist_last.reindex(co.index)
co["currently_listed"] = co.index.isin(currently_listed)

# Screen I_1: evidence of a national-exchange listing at some point
co["has_listing_evidence"] = co["currently_listed"] | co["first_8a12b"].notna() | co["last_delisting"].notna()

# list_start: 8-A12B date for companies new to the data (IPO-like), else first periodic filing
new_in_data = co["first_periodic"] > DATA_START + pd.Timedelta(days=365)
ipo_like = new_in_data & co["first_8a12b"].notna() & (co["first_periodic"] >= co["first_8a12b"] - pd.Timedelta(days=365))
co["list_start"] = co["first_8a12b"].where(ipo_like, co["first_periodic"])

# list_end: delisting form (if no longer listed) or end of reporting, whichever is first
end_delisting = co["last_delisting"].where(~co["currently_listed"])
end_reporting = co["last_periodic"] + ACTIVE_GRACE
co["list_end"] = pd.concat([end_delisting, end_reporting], axis=1).min(axis=1)
co.loc[co["list_end"] >= AS_OF, "list_end"] = pd.NaT          # still listed
co["status_unknown"] = ~co["currently_listed"] & co["last_delisting"].isna() & co["list_end"].isna()

# Master universe: listed at some point inside the window
master = co[co["has_listing_evidence"] & (co["list_end"].isna() | (co["list_end"] > WINDOW_START))].copy()
print(f"{len(co):,} periodic filers -> {len(master):,} with an exchange listing inside the window")

## Step 3: Company details
One EDGAR request per company returns its static details (name, SIC industry code, state of incorporation, business location, current tickers). This is the slow step: about 8 companies per second, so roughly 15-20 minutes on the first run. With `CACHE_DIR` set, later runs only fetch companies they haven't seen before.

In [0]:
def company_meta(cik):
    j = sec_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json", as_json=True)
    if not j:
        return {"cik": cik}
    biz = (j.get("addresses") or {}).get("business") or {}
    return {"cik": cik, "name": j.get("name"), "sic": j.get("sic"), "sic_desc": j.get("sicDescription"),
            "state_of_inc": j.get("stateOfIncorporation"), "business_location": biz.get("stateOrCountry"),
            "tickers": ",".join(j.get("tickers") or []),
            "exchanges": ",".join(e or "" for e in (j.get("exchanges") or []))}

meta_path = f"{CACHE_DIR}/company_meta.parquet" if CACHE_DIR else None
cached = pd.read_parquet(meta_path) if meta_path and os.path.exists(meta_path) else pd.DataFrame(columns=["cik"])
todo = sorted(set(master.index) - set(cached["cik"]))
print(f"Fetching details for {len(todo):,} companies (~{len(todo) / MAX_RPS / 60:.0f} min)")

t0 = time.time()
with cf.ThreadPoolExecutor(MAX_RPS) as ex:
    fetched = pd.DataFrame(list(ex.map(company_meta, todo)))
meta = pd.concat([cached, fetched], ignore_index=True).drop_duplicates("cik", keep="last")
meta["cik"] = meta["cik"].astype(int)
if meta_path:
    os.makedirs(CACHE_DIR, exist_ok=True)
    meta.to_parquet(meta_path)
meta = meta.set_index("cik")
print(f"Done in {time.time() - t0:.0f}s; {meta['sic'].isna().sum():,} companies without a SIC code")

## Step 4: Boolean screens \(I_k(c)\)
| Screen | Rule | Replaces (article) |
|---|---|---|
| `has_listing_evidence` | current Nasdaq / NYSE / Cboe ticker, 8-A12B, or Form 25 | eligible exchanges |
| `is_us_periodic_filer` | files 10-K / 10-Q (built into Step 2) | no ADRs / depositary shares |
| `is_operating_company` | SIC not in 6770 (SPACs), 6221 (commodity pools), 6189 (ABS) | security description screen |
| `is_known_status` | not `status_unknown` | ticker available |

The universe is **company-level** (CIK). Choosing the common-stock ticker or share class happens at the price step.

In [0]:
u = master.join(meta, how="left")
u["is_operating_company"] = ~u["sic"].isin(list(EXCLUDED_SIC))
u["is_known_status"] = ~u["status_unknown"]
u["is_in_starting_universe"] = u["is_operating_company"] & u["is_known_status"]

starting_universe = u.loc[u["is_in_starting_universe"],
    ["name", "sic", "sic_desc", "state_of_inc", "business_location", "tickers", "exchanges",
     "list_start", "list_end", "first_8a12b", "last_delisting", "last_periodic", "currently_listed"]].copy()

# Screen funnel
funnel = pd.DataFrame([
    {"step": "periodic filers (10-K / 10-Q) in data", "companies": len(co)},
    {"step": "+ exchange listing inside the window", "companies": len(master)},
    {"step": "+ operating company (SIC screen)", "companies": int(u["is_operating_company"].sum())},
    {"step": "+ known listing status", "companies": len(starting_universe)},
])
display(funnel)
display(u.loc[~u["is_operating_company"], "sic"].map(EXCLUDED_SIC).value_counts().rename_axis("excluded").reset_index(name="companies"))

## Step 5: Point-in-time universe
`universe_at(date)` returns the companies in the starting universe on that date. Every rebalance in later articles should call it, rather than using today's list.

In [0]:
def universe_at(d, su=starting_universe):
    """Companies in the starting universe on date d (list_start <= d < list_end)."""
    d = pd.Timestamp(d)
    alive = (su["list_start"] <= d) & (su["list_end"].isna() | (su["list_end"] > d))
    return su[alive]

quarter_ends = pd.date_range(WINDOW_START, AS_OF, freq="QE")
su = starting_universe
counts = pd.DataFrame({
    "date": quarter_ends,
    "companies": [len(universe_at(d)) for d in quarter_ends],
    "entered_in_quarter": [int(su["list_start"].between(d - pd.offsets.QuarterEnd(1), d, inclusive="right").sum()) for d in quarter_ends],
    "exited_in_quarter": [int(su["list_end"].between(d - pd.offsets.QuarterEnd(1), d, inclusive="right").sum()) for d in quarter_ends],
})
display(counts)

print(f"Today ({AS_OF.date()}): {len(universe_at(AS_OF)):,} companies")
print(f"Ever in the universe during the window: {len(su):,}; exited (delisted, acquired, stopped reporting): {su['list_end'].notna().sum():,}")

In [0]:
# 1) Today's slice vs today's ticker file (operating companies that file 10-K / 10-Q)
today_set = set(universe_at(AS_OF).index)
listed_now = set(u.index[u["currently_listed"] & u["is_operating_company"]])
print(f"In today's universe but no major-exchange ticker today: {len(today_set - listed_now):,} (expect 0)")
print(f"Major-exchange ticker today but not in today's universe: {len(listed_now - today_set):,} "
      "(expected few: stopped 10-K/10-Q filing, e.g. switched to 20-F)")

# 2) Spot checks on well-known listings and exits
spot = {320193: "Apple (listed throughout)", 1418091: "Twitter (taken private Oct 2022)",
        719739: "SVB Financial (failed Mar 2023)", 886158: "Bed Bath & Beyond (bankrupt 2023)",
        1559720: "Airbnb (IPO Dec 2020)", 1810806: "Unity Software (IPO Sep 2020)"}
display(starting_universe.reindex(list(spot))[["name", "list_start", "list_end", "last_delisting", "tickers"]]
        .assign(expected=list(spot.values())))

# 3) Exited companies without a ticker today: their price history will need a historical ticker
exited = su[su["list_end"].notna()]
print(f"Exited companies with no current ticker: {(exited['tickers'].fillna('') == '').sum():,} of {len(exited):,}")

In [0]:
# Persist for the next articles (rebalance schedule, prices). Overwritten on each refresh; the as_of column records the run date.
# Set OUTPUT_PATH = "" to skip saving.
OUTPUT_PATH = f"{DATA_DIR}/starting_universe.parquet"
if OUTPUT_PATH:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    starting_universe.assign(as_of=AS_OF, window_start=WINDOW_START).reset_index().to_parquet(OUTPUT_PATH)
    print(f"Saved {len(starting_universe):,} companies to {OUTPUT_PATH}")

## Known limitations
* **Listing dates are approximate.**
  * A Form 25 takes effect about 10 days after it's filed.
  * The 200-day reporting grace can keep a company in the universe a few months after it actually stopped trading.
  * Companies that moved up from OTC to an exchange inside the window count as listed from their first filing.
* **Form 25 can cover other securities.** A Form 25 may be for notes or preferreds rather than the common stock. It only ends a company's listing if the company has no major-exchange ticker today.
* **Company-level, not security-level.** Share-class selection (e.g. GOOGL vs GOOG) and the historical ticker for exited companies are resolved at the price step.
* **Some survivorship bias can come back at the price step.** Stocks with no public price history are skipped (TPP rule). Most of them will be exited companies, so count and report what gets skipped each year.
* **Excluded vs the article:**
  * Foreign 20-F / 40-F filers are excluded, including foreign ordinary shares the article kept.
  * Business development companies (BDCs) and closed-end funds that file 10-Ks are not excluded by the SIC screen.